In [ ]:
import pandas as pd
import numpy as np
import sys
import yaml, optuna, torch

from hdbscan import HDBSCAN
sys.path.append("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/LatentGEE")
from LatentGEE import *

# Data

In [ ]:
# hivrc_filtered = np.load("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/hivrc_filtered_expm1.npy", allow_pickle=True)
hivrc_filtered = pd.read_pickle("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/LatentGEE/Data/hivrc_filtered_expm1.pkl")
# X_tensor = torch.from_numpy(hivrc_filtered).float()

X_tensor = torch.load("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/LatentGEE/Data/hivrc_tensor_expm1.pt")

In [ ]:
hivrc_filtered.shape

In [ ]:
# LatentGEE Implementation

import yaml
import numpy as np
import pandas as pd
import seaborn as sns
import tqdm
import optuna

import torch
import torch.nn as nn
import torch.nn.functional as F
from functools import partial

from hdbscan import HDBSCAN            # <— density-based clustering
from sklearn.decomposition import PCA
from sklearn.manifold import MDS
from sklearn.metrics import silhouette_score, pairwise_distances
from sklearn.preprocessing import StandardScaler

from skbio.stats.distance import DistanceMatrix, permanova
from skbio.stats.composition import clr, multiplicative_replacement

from statsmodels.genmod.generalized_estimating_equations import GEE
from statsmodels.genmod.cov_struct import Exchangeable
from statsmodels.genmod.families import Gaussian

import matplotlib.pyplot as plt
from matplotlib.patches import Ellipse

from scipy.spatial.distance import pdist, squareform


# 베스트 모델 저장/재현
def save_model(model, path="best_model.pt"):
    torch.save(model.state_dict(), path)

def load_model(model_class, path="best_model.pt", *args, **kwargs):
    model=model_class(*args, **kwargs)
    model.load_state_dict(torch.load(path))
    model.eval()
    return model

# Toy example with >= 1000 samples
def simulate_microbiome_data(n_samples=1000, n_features=50, n_batches=3):
    np.random.seed(42)
    batch_labels = np.random.choice(range(n_batches), size=n_samples)
    data = []
    for i in range(n_samples):
        base=np.random.lognormal(mean=1, sigma=1, size=n_features)
        if batch_labels[i]==1:
            base[:10] += 2.0
        elif batch_labels[i]==2:
            base[10:20]+=1.5
        mask = np.random.binomial(1, 0.3, size=n_features)
        data.append(base*mask)
    return np.array(data), batch_labels

def decode_batch_corrected_latent(model, z_resid, save_path="X_corrected.npy"):
    """
    Decode batch-corrected latent z to reconstructed X, and save to file.
    
    Args:
        model: trained VAE model (with .decode() method)
        z_resid: numpy array of batch-corrected latent representations (z~)
        save_path: file path to save the reconstructed X (as .npy)
    
    Returns:
        x_corrected: decoded reconstruction of shape (n_samples, n_features)
    """
    model.eval()
    z_resid_tensor = torch.tensor(z_resid, dtype=torch.float32)
    with torch.no_grad():
        pi, mu, log_sigma = model.decode(z_resid_tensor)
        x_corrected = mu.cpu().numpy()
    np.save(save_path, x_corrected)
    return x_corrected


# Flexible MLP builder with dropout and activation selection
class FlexibleMLP(nn.Module):
    def __init__(self, layer_dims, dropout_rate=0.0, activation='relu'):
        super().__init__()
        layers = []
        act_fn = self.get_activation_fn(activation)
        for i in range(len(layer_dims) - 1):
            layers.append(nn.Linear(layer_dims[i], layer_dims[i + 1]))
            if i < len(layer_dims) - 2:
                layers.append(act_fn)
                if dropout_rate > 0.0:
                    layers.append(nn.Dropout(dropout_rate))
        self.net = nn.Sequential(*layers)
        
    def get_activation_fn(self, name):
        if name == 'relu':
            return nn.ReLU()
        elif name == 'leaky_relu':
            return nn.LeakyReLU(0.2)
        elif name == 'gelu':
            return nn.GELU()
        else:
            raise ValueError(f"Unsupported activation function: {name}")

    def forward(self, x):
        return self.net(x)

# Network architecture generator
def build_layer_dims(input_dim, output_dim, n_layers, base_dim, strategy='constant'):
    dims = [input_dim]
    current_dim = base_dim
    for _ in range(n_layers):
        dims.append(current_dim)
        if strategy == 'halve':
            current_dim = max(current_dim // 2, output_dim)
        elif strategy == 'double':
            current_dim = min(current_dim * 2, 1024)  # arbitrary upper limit
        elif strategy == 'constant':
            continue
        else:
            raise ValueError(f"Unknown strategy: {strategy}")
    dims.append(output_dim)
    return dims

# LatentGEE-U VAE using flexible encoder/decoder
class VAE(nn.Module):
    def __init__(self, input_dim, latent_dim, n_layers, base_dim,
                 strategy='constant', dropout_rate=0.0, activation='relu'):
        super().__init__()
        # ---------- Encoder ----------
        enc_dims = build_layer_dims(input_dim, base_dim, n_layers, latent_dim, strategy)
        self.enc_net = FlexibleMLP(enc_dims, dropout_rate, activation)
        self.fc_mu     = nn.Linear(enc_dims[-1], latent_dim)
        self.fc_logvar = nn.Linear(enc_dims[-1], latent_dim)

        # ---------- Decoder ----------
        dec_dims = build_layer_dims(latent_dim, base_dim, n_layers, input_dim, strategy)
        self.dec_net   = FlexibleMLP(dec_dims, dropout_rate, activation)
        self.dec_pi    = nn.Linear(dec_dims[-1], input_dim)   # zero prob
        self.dec_mu    = nn.Linear(dec_dims[-1], input_dim)   # log-normal μ
        self.dec_log_sigma  = nn.Linear(dec_dims[-1], input_dim)   # log-normal _sigma

    def encode(self, x):
        h = self.enc_net(x)
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h  = self.dec_net(z)
        pi = torch.sigmoid(self.dec_pi(h))          # (0 ~ 1)
        mu = self.dec_mu(h)                         # 실수
        log_sigma = self.dec_log_sigma(h)                     # 실수
        return pi, mu, log_sigma    # 3-tuple 반환

    def forward(self, x):
        mu_z, logvar_z = self.encode(x)
        z   = self.reparameterize(mu_z, logvar_z)
        pi, mu_x, log_sigma_x = self.decode(z)
        return (pi, mu_x, log_sigma_x), mu_z, logvar_z, z
# --------------------------------------------------
# 2. ZILN 음의 log-우도 함수
# --------------------------------------------------
def ziln_nll(x, pi, mu, log_sigma, eps=1e-8):
    """
    x      : (N, D)  원 데이터 (양수 또는 0)
    pi     : P(x==0)  (N, D)
    mu     : 로그-정규 평균
    log_sigma   : 로그-정규 log-std
    """
    # ① x == 0 부분
    nll_zero = -(x == 0).float() * torch.log(pi + eps)

    # ② x > 0 부분 (LogNormal pdf)
    pos_mask = (x > 0).float()
    log_x    = torch.log(x + eps)
    log_pdf  = -0.5 * (((log_x - mu) / log_sigma.exp()) ** 2) \
               - log_sigma - 0.5 * np.log(2 * np.pi)           # log-pdf(로그정규)
    nll_pos  = -pos_mask * (torch.log(1 - pi + eps) + log_pdf)

    return (nll_zero + nll_pos).mean()      # mean over samples & features

# --------------------------------------------------
# 3. Pseudo-batch clustering 함수
# --------------------------------------------------
def pseudo_clustering(z_tensor, 
                      min_cluster_size: int = 10,
                      min_samples: int | None = None,
                      metric="euclidean",
                      cluster_selection_method="eom"):
    """
    ▣ 역할
        1. HDBSCAN → pseudo-batch 라벨 생성
    """
    if z_tensor.requires_grad:
        z_tensor = z_tensor.detach()
    z_cpu = z_tensor.cpu().numpy()

    # ── 1. Pseudo-batch 탐색 ─────────────────────────────
    hdb = HDBSCAN(
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric=metric,
        cluster_selection_method=cluster_selection_method
    )
    labels = hdb.fit_predict(z_cpu)          # shape (N,)    
    return labels

# --------------------------------------------------
# 4. gee latent residualizaiton
# --------------------------------------------------
def gee_latent_residual(z_np, 
                        pseudo_batch_labels, 
                        covariates_df=None):

    df = pd.DataFrame(z_np, columns=[f"z{i}" for i in range(z_np.shape[1])])
    df["cluster"] = pseudo_batch_labels

    cov_names = []

    if covariates_df is not None:

        if isinstance(covariates_df, pd.DataFrame):

            df = pd.concat([df, covariates_df], axis=1)
            cov_names = list(covariates_df.columns)

        elif isinstance(covariates_df, np.ndarray):

            for i in range(covariates_df.shape[1]):
                name = f"cov{i}"
                df[name] = covariates_df[:, i]
                cov_names.append(name)

    residuals = []

    latent_cols = [c for c in df.columns if c.startswith("z")]

    for col in latent_cols:

        if covariates_df is None:
            formula = f"{col} ~ 1"
        else:
            formula = f"{col} ~ {' + '.join(cov_names)}"

        model = GEE.from_formula(
            formula,
            groups="cluster",
            data=df,
            family=Gaussian(),
            cov_struct=Exchangeable()
        )

        result = model.fit()

        resid = df[col] - result.fittedvalues

        residuals.append(resid.values)

    return np.vstack(residuals).T

def evaluate_latentgee(
        model: nn.Module,
        X_tensor: torch.Tensor,
        covariates_df: pd.DataFrame | None = None,
        min_cluster_size: int = 10,
        min_samples: int | None = None,
        metric: str = "euclidean",
    ) -> tuple[np.ndarray, float]:

    """
    LatentGEE evaluation

    Steps
    -----
    1. Encode data to latent space
    2. HDBSCAN clustering → pseudo-batch
    3. GEE residualization
    4. silhouette score calculation
    """

    model.eval()

    with torch.no_grad():
        mu, logvar = model.encode(X_tensor)
        z = model.reparameterize(mu, logvar)
        z_np = z.cpu().numpy()

    # ---- pseudo-batch clustering ----
    labels = pseudo_clustering(
        mu,
        min_cluster_size=min_cluster_size,
        min_samples=min_samples,
        metric = metric
    )

    # ---- remove noise samples ----
    mask = labels != -1

    n_valid = np.unique(labels[mask]).size
    if n_valid < 2 or mask.sum() < 3:
        raise optuna.TrialPruned()

    z_used = z_np[mask]
    lbl_used = labels[mask]
    noise_ratio = (labels == -1).mean()

    # ---- covariate subset ----
    if covariates_df is not None:
        cov_used = covariates_df.iloc[mask]
    else:
        cov_used = None

    # ---- GEE residualization ----
    z_resid = gee_latent_residual(
        z_used,
        lbl_used,
        covariates_df=cov_used
    )

    # ---- silhouette score ----
    sil = silhouette_score(z_resid, lbl_used)

    return labels, sil, noise_ratio
    
# --------------------------------------------------
# 3. train_vae() 에서 ZILN 손실 사용
# --------------------------------------------------
def train_vae(model, data_tensor, epochs=50, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    for ep in range(epochs):
        model.train()
        (pi, mu_x, log_sigma_x), mu_z, logvar_z, _ = model(data_tensor)

        recon_nll = ziln_nll(data_tensor, pi, mu_x, log_sigma_x) # negative log likelihood 
        kl = -0.5 * torch.mean(1 + logvar_z - mu_z.pow(2) - logvar_z.exp())
        loss = recon_nll + kl

        if torch.isnan(loss):
            raise optuna.TrialPruned()

        opt.zero_grad(); loss.backward(); opt.step()
        if (ep + 1) % 10 == 0:
            print(f"[{ep+1}/{epochs}] ZILN-NLL {recon_nll:.4f}  KL {kl:.4f}")
    return loss.item()


def permanova_r2(
    X: np.ndarray,
    grouping,
    metric:str = "braycurtis",
    ids=None,
    permutations:int = 999,
    pseudocount:float = 1e-6,
):
    """
    Calculate PERMANOVA (pseudo-F, p-value) and R² like vegan::adonis2.
    
    Parameters
    ----------
    X : (n_samples, n_features) array_like
        Feature (count/proportion) matrix.
    grouping : 1-D array_like
        Group labels, length == n_samples.
    metric : str, default 'braycurtis'
        Distance metric for `pdist`.  
        Special case: 'aitchison' ⇒ clr-Euclidean.
    ids : list[str] or None
        Sample IDs for the distance matrix.
    permutations : int
        Number of permutations for PERMANOVA.
    pseudocount : float
        Added to zeros before clr, if metric == 'aitchison'.
    """
    X = np.asarray(X.astype(float))
    grouping = np.asarray(grouping)

    # ---------- sample IDs ----------
    if ids is None:
        ids = [f"S{i+1}" for i in range(X.shape[0])]

    # ---------- distance matrix ----------
    if metric.lower() == "aitchison":
        # 1) zero replacement → clr transform
        X_clr = clr(multiplicative_replacement(X, pseudocount))
        # 2) Euclidean distance in clr space == Aitchison distance
        dist = squareform(pdist(X_clr, metric="euclidean"))
    else:
        dist = squareform(pdist(X, metric=metric))

    dm = DistanceMatrix(dist, ids=ids)

    # ---------- R² ----------
    d2 = dm.data ** 2
    n = d2.shape[0]
    sst = d2[np.triu_indices(n, 1)].sum() / n

    ssw = 0.0
    for g in np.unique(grouping):
        idx = np.where(grouping == g)[0]
        if len(idx) < 2:
            continue
        d2_g = d2[np.ix_(idx, idx)]
        ssw += d2_g[np.triu_indices(len(idx), 1)].sum() / len(idx)

    ssb = sst - ssw
    r2 = ssb / sst

    # ---------- PERMANOVA ----------
    res = permanova(dm, grouping, permutations=permutations)

    return res, r2


# objective

In [ ]:
# ---------------------------------------------------------
# 1. helper: base_dim 선택 (원하는 경우만 사용)
def choose_base_dim(input_dim: int, strategy: str, n_layers: int) -> int:
    """
    • 'halve'  : input_dim // 2
    • 'double' : input_dim // 4  (폭 좁혀 시작)
    • default  : input_dim // 4
    """
    if strategy == "halve":
        return max(input_dim // 2, 8)
    elif strategy == "double":
        return max(input_dim // 4, 8)
    else:                   # 'constant'
        return max(input_dim // 4, 8)
# ---------------------------------------------------------
# 2. Optuna objective
def objective(trial: optuna.Trial,
              config: dict,
              X_tensor: torch.Tensor,
              log_file: str = "optuna_trial_log.csv") -> float:
    input_dim = X_tensor.shape[1]

    # ── ① 파라미터 샘플링 ─────────────────────────
    strategy      = trial.suggest_categorical("strategy",
                        config["search_space"]["model"]["strategy"])
    n_layers      = trial.suggest_int("n_layers",
                        *config["search_space"]["model"]["n_layers"])
    base_dim = trial.suggest_categorical("base_dim",
                                         config["search_space"]["model"]["base_dim"])
    latent_dim    = trial.suggest_int("latent_dim",
                        *config["search_space"]["model"]["latent_dim"])
    activation    = trial.suggest_categorical("activation",
                        config["search_space"]["model"]["activation"])
    dropout_rate  = trial.suggest_float("dropout_rate",
                        *config["search_space"]["model"]["dropout_rate"])
    epochs        = trial.suggest_categorical("epochs",
                        config["search_space"]["training"]["epochs"])
    
    lr_low, lr_high = sorted(map(float,config["search_space"]["training"]["learning_rate"]["loguniform"]))
    learning_rate = trial.suggest_float("learning_rate", lr_low, lr_high, log=True)

    # HDBSCAN
    mcs_low, mcs_high   = config["search_space"]["clustering"]["min_cluster_size"]
    min_cluster_size    = trial.suggest_int("min_cluster_size", mcs_low, mcs_high)
    min_samples_token   = trial.suggest_categorical("min_samples_token",config["search_space"]["clustering"]["min_samples"])
    min_samples         = None if (min_samples_token in ["None", None]) else int(min_samples_token)
    metric              = trial.suggest_categorical("metric", config["search_space"]["clustering"]["metric"])


    # ── Device 설정 ──────────────────────── 
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # ── ② 모델 구성 & 학습 ──────────────────────── 

    model = VAE(input_dim=input_dim,
                latent_dim=latent_dim,
                n_layers=n_layers,
                base_dim=base_dim,
                strategy=strategy,
                dropout_rate=dropout_rate,
                activation=activation).to(device)
    
    X_tensor = X_tensor.to(device)
    try:
        last_loss = train_vae(model, X_tensor, epochs=epochs, lr=learning_rate)
        last_loss = float(last_loss)           # Tensor → python scalar
    
    except RuntimeError as e:
        if "out of memory" in str(e):
            raise optuna.TrialPruned()
        else:
            raise

    # ── ③ 평가 ───────────────────────────────────
    try:
        labels, score, noise_ratio = evaluate_latentgee(
            model, X_tensor,
            min_cluster_size=min_cluster_size,
            min_samples=min_samples,
            metric=metric
        )
    except ValueError:                         # 실루엣 계산 실패
        raise optuna.TrialPruned()

    if np.isnan(score) or np.isinf(score):
        raise optuna.TrialPruned()

    # ── ④ 로그 ───────────────────────────────────
    n_clusters = len(np.unique(labels[labels != -1]))
    res_df = pd.DataFrame({
        "trial_number":  [trial.number],
        "base_dim":      [base_dim],
        "n_layers":      [n_layers],
        "latent_dim":    [latent_dim],
        "activation":    [activation],
        "strategy":      [strategy],
        "dropout_rate":  [dropout_rate],
        "epochs":        [epochs],
        "learning_rate": [learning_rate],
        "loss":          [last_loss],
        "silhouette":    [score],
        "n_clusters":    [n_clusters],
        "noise_ratio":   [noise_ratio]
    })
    mode, header = ("w", True) if trial.number == 0 else ("a", False)
    res_df.to_csv(log_file, mode=mode, index=False, header=header)

    print(f"Trial {trial.number:3d} | sil={score:+.4f} | k={n_clusters}")

    # ── ⑤ Optuna가 최대화할 스코어 반환 ───────────
    return score

In [ ]:
# Optuna Optimazation(n_trials=1000)

In [ ]:
# ---- 1. YAML 로드 ----
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

# ---- 2. Optuna 실행 ----
study = optuna.create_study(direction="maximize")
study.optimize(
    lambda tr: objective(tr, cfg, X_tensor),   # ← 기존 objective 그대로
    n_trials=1000,
)

In [ ]:
# study 객체는 이미 메모리에 있다고 가정
best_trial   = study.best_trial
best_params  = best_trial.params
print("Best hyper-parameters\n", best_params)

# ── 2-1. 파라미터 꺼내기 ───────────────────────────
input_dim    = X_tensor.shape[1]
latent_dim   = best_params["latent_dim"]
n_layers     = best_params["n_layers"]
strategy     = best_params["strategy"]
dropout_rate = best_params["dropout_rate"]
activation   = best_params["activation"]
epochs       = best_params["epochs"]          # 30 · 50 · 80 · 100 중 하나
lr           = best_params["learning_rate"]
min_cs       = best_params["min_cluster_size"]
min_samples  = None                           # search space가 "None" 뿐이므로

# ── 2-2. base_dim 결정 규칙 (같은 함수 재사용) ──────
base_dim = choose_base_dim(input_dim, strategy, n_layers)

# ── 2-3. 모델 인스턴스 ─────────────────────────────
best_model = VAE(
    input_dim   = input_dim,
    latent_dim  = latent_dim,
    n_layers    = n_layers,
    base_dim    = base_dim,
    strategy    = strategy,
    dropout_rate= dropout_rate,
    activation  = activation,
)

# 2) encode → reparameterize → decode
best_model.eval()
with torch.no_grad():
    # a) get μ, logvar
    mu_z, logvar_z = best_model.encode(X_tensor)
    # b) reparameterize
    z = best_model.reparameterize(mu_z, logvar_z)
    # c) decode
    pi, mu_x, log_sigma_x = best_model.decode(z)
    sigma_x = torch.exp(log_sigma_x)
    # d) compute E[X] under ZILN: (1−π) · exp(μ + ½_sigma²)
    X_recon = (1 - pi) * torch.exp(mu_x + 0.5 * sigma_x**2)

# 3) move back to CPU / numpy
recon_np = X_recon.cpu().numpy()

# 4) wrap in a DataFrame with original sample / feature labels
recon_df = pd.DataFrame(
    recon_np,
    index=hivrc_filtered.index,      # same sample order
    columns=hivrc_filtered.columns   # same OTU names
)

# 5) save to disk
recon_df.to_csv("X_reconstructed_ZILN.csv")

print("Reconstruction complete — written to X_reconstructed_ZILN.csv")
# ─── 1. 저장 ───────────────────────────────────────────
save_path = "best_model_0618.pt"
torch.save(best_model.state_dict(), save_path)
print(f"✓ 모델 가중치를 {save_path} 로 저장했습니다.")


In [ ]:
recon_df

In [ ]:
best_model.eval()
with torch.no_grad():
    mu, logvar = best_model.encode(X_tensor)      # (N, k)
    z          = mu                               # re-parameterisation 없이 μ 사용
    # ── GEE residual로 batch effect 제거 ────────────
    z_tilde, _ = gee_residual(z, min_cs, min_samples)  # 직접 만든 함수
    # ── 디코더로 재구성 ─────────────────────────────
    X_recon    = best_model.decode(z_tilde)       # (N, D)Í›

In [ ]:
from sklearn.manifold import MDS
import matplotlib.pyplot as plt




mds = MDS(n_components=2, random_state=0)
mds_raw  = mds.fit_transform(hivrc_filtered)
mds_corr = mds.fit_transform(recon_df)
study_codes = pd.Categorical(hivrc_merged['Study']).codes

plt.scatter(mds_raw[:,0], mds_raw[:,1], c=study_codes, cmap='tab10', alpha=0.3, label='raw')
plt.scatter(mds_corr[:,0], mds_corr[:,1], c=study_codes, cmap='tab10', alpha=0.7, label='corrected')
plt.legend(); plt.title("Raw vs Corrected MDS Overlay")
plt.show()

In [ ]:
hivrc_filtered = pd.read_pickle("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/hivrc_filtered_expm1.pkl")
X_reconstructed = pd.read_csv("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/X_reconstructed_ZILN.csv")
hivrc_meta = pd.read_excel(
    "/Users/linayang/Downloads/slyang/research/Data/Metagenome/externaldata/syn18406854/SupplementaryMaterial.xlsx",
    header=1,  # 두 번째 행을 컬럼명으로 사용
    usecols="B:F"  # 첫 번째 열(A열)은 제외
)
hivrc_merged = pd.merge(hivrc_filtered.reset_index(), hivrc_meta[["SeqID","Study"]], on="SeqID", how="inner")
X_reconstructed_merged = pd.merge(X_reconstructed, hivrc_meta[["SeqID","Study"]], on="SeqID", how="inner")



hivrc_merged.set_index('SeqID', inplace=True)
X_reconstructed_merged.set_index('SeqID', inplace=True)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12,6), sharex=True, sharey=True)

# Raw
sc = axes[0].scatter(
    mds_raw[:,0], mds_raw[:,1],
    c=study_codes, cmap='tab10', alpha=0.7
)
axes[0].set_title("Raw")
# Corrected
axes[1].scatter(
    mds_corr[:,0], mds_corr[:,1],
    c=study_codes, cmap='tab10', alpha=0.7
)
axes[1].set_title("Corrected")

# 전체 범례
fig.legend(*sc.legend_elements(), title="Study", loc="upper right")
plt.suptitle("Raw vs Corrected MDS (side-by-side)")
plt.show()

# hivrc_filtered_0619

In [ ]:
hivrc_filtered_0619 = pd.read_pickle("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/hivrc_filtered_0619.pkl")
# X_tensor = torch.from_numpy(hivrc_filtered).float()

X_tensor_0619 = torch.load("/Users/linayang/Documents/연구 관련 문서/졸업 후 연구/hivrc_tensor_0619.pt")

In [ ]:
# ---- 1. YAML 로드 ----
with open("config.yaml") as f:
    cfg = yaml.safe_load(f)

# ---- 2. Optuna 실행 ----
study = optuna.create_study(direction="maximize")
study.optimize(
    lambda tr: objective(tr, cfg, X_tensor_0619),   # ← 기존 objective 그대로
    n_trials=1000,
)

In [ ]:
# study 객체는 이미 메모리에 있다고 가정
best_trial   = study.best_trial
best_params  = best_trial.params
print("Best hyper-parameters\n", best_params)

# ── 2-1. 파라미터 꺼내기 ───────────────────────────
input_dim    = X_tensor_0619.shape[1]
latent_dim   = best_params["latent_dim"]
n_layers     = best_params["n_layers"]
strategy     = best_params["strategy"]
dropout_rate = best_params["dropout_rate"]
activation   = best_params["activation"]
epochs       = best_params["epochs"]          # 30 · 50 · 80 · 100 중 하나
lr           = best_params["learning_rate"]
min_cs       = best_params["min_cluster_size"]
min_samples  = None                           # search space가 "None" 뿐이므로

# ── 2-2. base_dim 결정 규칙 (같은 함수 재사용) ──────
base_dim = choose_base_dim(input_dim, strategy, n_layers)
# ── 2-3. 모델 인스턴스 ─────────────────────────────
best_model = VAE(
    input_dim   = input_dim,
    latent_dim  = latent_dim,
    n_layers    = n_layers,
    base_dim    = base_dim,
    strategy    = strategy,
    dropout_rate= dropout_rate,
    activation  = activation,
)

# 2) encode → reparameterize → decode
best_model.eval()
with torch.no_grad():
    # a) get μ, logvar
    mu_z, logvar_z = best_model.encode(X_tensor_0619)
    # b) reparameterize
    z = best_model.reparameterize(mu_z, logvar_z)
    # c) decode
    pi, mu_x, log_sigma_x = best_model.decode(z)
    sigma_x = torch.exp(log_sigma_x)
    # d) compute E[X] under ZILN: (1−π) · exp(μ + ½_sigma²)
    X_recon = (1 - pi) * torch.exp(mu_x + 0.5 * sigma_x**2)


# 3) move back to CPU / numpy
recon_np = X_tensor_0619.cpu().numpy()

# 4) wrap in a DataFrame with original sample / feature labels
recon_df = pd.DataFrame(
    recon_np,
    index=hivrc_filtered_0619.index,      # same sample order
    columns=hivrc_filtered_0619.columns   # same OTU names
)

# 5) save to disk
recon_df.to_csv("X_reconstructed_ZILN(0621).csv")

print("Reconstruction complete — written to X_reconstructed_ZILN(0621).csv")
# ─── 1. 저장 ───────────────────────────────────────────
# save_path = "best_model_0621.pt"
# torch.save(best_model.state_dict(), save_path)
print(f"✓ 모델 가중치를 {save_path} 로 저장했습니다.")
torch.save({
    'model_state_dict': best_model.state_dict(),
    'params': best_params  # ← best_trial.params 저장!
}, save_path)
print(f"✓ 모델 가중치를 {save_path} 로 저장했습니다.")


In [ ]:
X_tensor_0619.shape

In [ ]:
# import torch

# 저장된 모델 불러오기
best_model_0621 = torch.load('best_model_with_params.pt')
# best_trial.params 불러오기
best_params = best_model_0621['params']
state_dict  = best_model_0621['model_state_dict']
# 모델 구성에 필요한 정보
input_dim = X_tensor_0619.shape[1]  # raw input의 feature 수

# 필요한 파라미터 추출
latent_dim   = best_params["latent_dim"]
n_layers     = best_params["n_layers"]
strategy     = best_params["strategy"]
dropout_rate = best_params["dropout_rate"]
activation   = best_params["activation"]
base_dim     = choose_base_dim(input_dim, strategy, n_layers)

# 모델 인스턴스 생성
best_model_0621 = VAE(
    input_dim=input_dim,
    latent_dim=latent_dim,
    n_layers=n_layers,
    base_dim=base_dim,
    strategy=strategy,
    dropout_rate=dropout_rate,
    activation=activation,
)

# 저장된 파라미터 로드
best_model_0621.load_state_dict(state_dict)
best_model_0621.eval()


with torch.no_grad():
    # a) get μ, logvar
    mu_z, logvar_z = best_model_0621.encode(X_tensor_0619)
    # b) reparameterize
    z = best_model.reparameterize(mu_z, logvar_z)
    # c) decode
    pi, mu_x, log_sigma_x = best_model_0621.decode(z)
    sigma_x = torch.exp(log_sigma_x)
    # d) compute E[X] under ZILN: (1−π) · exp(μ + ½sigma²)
    X_recon_tmp = (1 - pi) * torch.exp(mu_x + 0.5 * sigma_x**2)

# numpy로 변환
recon_np_tmp = X_recon_tmp.cpu().numpy()

# DataFrame으로 정리 (옵션)
recon_df = pd.DataFrame(recon_np_tmp, 
                        index=hivrc_filtered_0619.index, 
                        columns=hivrc_filtered_0619.columns)

In [ ]:
recon_np_tmp